In [1]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

INPUT_DIR = Path("input_test")


def load_documents():
    documents = []

    for file_path in INPUT_DIR.rglob("*"):
        if file_path.suffix.lower() not in [".md", ".txt"]:
            continue

        text = file_path.read_text(encoding="utf-8")

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": str(file_path),
                    "filename": file_path.name,
                }
            )
        )

    return documents


documents = load_documents()

normal_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=500,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

normal_chunks = normal_splitter.split_documents(documents)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)


sem_chunks = []

for document in documents:

    chunks = semantic_splitter.split_documents([document])

    sem_chunks.extend(chunks)

def hybrid_chunk(document):

    semantic_chunks = semantic_splitter.split_documents(
        [document]
    )

    final_chunks = []

    for semantic_chunk in semantic_chunks:

        # If semantic chunk is reasonably sized,
        # keep it intact.
        if len(semantic_chunk.page_content) <= 4000:

            final_chunks.append(
                semantic_chunk
            )

        else:

            # Semantic chunk is too large.
            # Split it using fixed-size chunking.

            smaller_chunks = normal_splitter.split_documents(
                [semantic_chunk]
            )

            final_chunks.extend(
                smaller_chunks
            )

    return final_chunks


hybrid_chunks = []

for document in documents:

    chunks = hybrid_chunk(document)

    hybrid_chunks.extend(chunks)

/tmp/ipykernel_61092/1779389591.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [2]:
normal_chunks[:5]
sem_chunks[:5]
hybrid_chunks[:5]

[Document(metadata={'source': 'input_test/Confined_Space_Entry_Procedure.md', 'filename': 'Confined_Space_Entry_Procedure.md'}, page_content='<table><tr><td></td><td>HEALTH &amp; SAFETY POLICY/PROCEDURE</td><td></td></tr><tr><td colspan="3">CONFINED SPACE ENTRY</td></tr></table>\n\n## Purpose\n\nThis Code of practice outlines what is necessary to ensure the Safety & Health of all individuals at all times while entering and working in a confined space. This includes the responsibilities of all parties involved to ensure proper confined space identification, preparation, authorization and communication before and during the confined space entry and confined space work. ## Scope\n\nThis Code of Practice is applicable to all Company Name personnel, contractors and any other groups providing services at a Company Name worksite. ## Definitions\n\n## Confined Space\n\nA “confined space" is any work area which is not intended for continuous worker occupancy, and has, by design, structure, loca

In [3]:
from langchain_chroma import Chroma

PER_DIR = './vector_db'
normal_db = Chroma(
                embedding_function=embeddings,
                persist_directory=PER_DIR,
                collection_name='normal_collection',
                )
normal_db.add_documents(normal_chunks)

semantic_db = Chroma(
                embedding_function=embeddings,
                persist_directory=PER_DIR,
                collection_name='semantic_collection',
                )
semantic_db.add_documents(sem_chunks)

hybrid_db = Chroma(
                embedding_function=embeddings,
                persist_directory=PER_DIR,
                collection_name='hybrid_collection',
                )
hybrid_db.add_documents(hybrid_chunks)

['ee0ba94e-304d-400b-be8d-ab697e3005f3',
 '4a604404-c3b3-43d3-99d7-01e1d5c9ac07',
 '8f390af8-d387-421c-9090-4c0af6b98b88',
 'a8270685-ab27-497a-9bc1-a7a6f2f452a7',
 '8e98ff0a-12b6-45cb-a30e-9f7dbffbd234',
 '44002466-b199-42d9-a9ef-0af755672b48',
 '12885fef-f139-457f-9004-45c902ea59e5',
 '46107ba9-ece1-42b6-8801-e4fe7b282c50',
 '567472ea-2a39-47cf-8d29-86893c8d4745',
 '8da720a2-49e4-40ea-a181-5849a813deed',
 '08593259-473c-4d15-8af0-42c637f6b69e',
 'c00f1957-7d7e-4def-8be1-1c711e70fa22',
 '737700e1-5752-43cb-94a9-6d9a9c858295',
 'bdd469ad-7613-42fb-8214-bfae9e16fb98',
 '05ebc8d5-0dec-439c-b753-ee807c90d721',
 'd1425199-7ce4-47ad-8eba-f840734fce99',
 'ea8fa3e7-1c57-497e-815c-2518daa78cac',
 '65ea5fd8-980e-4250-abbb-399549c8855c',
 '8c8372cb-25b7-475c-98d7-75aedc98e3ad',
 '195dabc2-0696-45c1-b7c1-fce4d0cf3a01',
 '1efcf024-dff8-4f15-80d2-2b9ac2289066',
 'd1839c60-2e1d-4de3-ae89-38e5419d30db',
 '2e59afd1-2840-4d04-9d06-2c93ced751f9',
 'b22c031f-77f2-4733-a99b-b9d1c24f0d2c',
 '83701820-f7b2-

In [4]:
print("Fixed chunks:", normal_db._collection.count())
print("Semantic chunks:", semantic_db._collection.count())
print("Hybrid chunks:", hybrid_db._collection.count())

Fixed chunks: 132
Semantic chunks: 81
Hybrid chunks: 189


In [6]:
query = "What is the approximate heat removal capacity of the MCS?"


fixed_results = normal_db.similarity_search(
    query,
    k=5
)

semantic_results = semantic_db.similarity_search(
    query,
    k=5
)

hybrid_results = hybrid_db.similarity_search(
    query,
    k=5
)

def print_results(name, results):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    for i, doc in enumerate(results):

        print(f"\n--- Result {i + 1} ---")

        print("Source:", doc.metadata.get("source"))

        print("Chunk ID:", doc.metadata.get("chunk_id"))

        print("\n", doc.page_content[:1000])


print_results("FIXED", fixed_results)

print_results("SEMANTIC", semantic_results)

print_results("HYBRID", hybrid_results)


FIXED

--- Result 1 ---
Source: input_test/20053810.md
Chunk ID: None

 # HEAT TRANSPORTSYSTEM SHUTDOWNOPERATION

## OBJECTIVES:

After completing this module you will be able to:

8.1 Explain the reason why a heat removal path must be available when the unit is shut down.

8.2 Explain the reason for the class of power supply provided for the direct and indirect shutdown cooling systems.

8.3 Explain the operation of the two current types of shutdown cooling systems used in CANDU reactors, ie, indirect and direct

8.4 State the function of the: a) Shutdown Cooling System, b) Maintenance Cooling System.

8.5 Expiain the need for a maintenance cooling system when the method of shutdown cooling is indirect.

8.6 For each of the operating states listed below, label a block diagram that shows the role of the HT system in transferring heat energy (major pathway only) from the heat source to the heat sink.

a) Zero power hot, pressurized when using a direct shutdown cooling system,

b) Zero 